## Validation of the Evaluation Dataset

Before starting the system comparison and evaluation, the saved evaluation dataset must be checked to ensure that it is complete and correctly structured.

This validation checks:
- The total number of evaluation questions.
- The distribution of questions across the four categories.
- The distribution of answerable and unanswerable questions.
- Duplicate question IDs.
- Missing values.

This step ensures that the evaluation dataset is ready for the next stage of the project.

In [12]:
import pandas as pd
from pathlib import Path

# Load the saved evaluation dataset
evaluation_df = pd.read_csv(
    "evaluation/evaluation_dataset_80_questions.csv"
)

# Check the total number of questions
print("Total questions:", len(evaluation_df))

# Check the number of questions in each category
print("\nQuestions by category:")
print(evaluation_df["category"].value_counts())

# Check the answerable and unanswerable distribution
print("\nAnswerable distribution:")
print(evaluation_df["answerable"].value_counts())

# Check for duplicate question IDs
print(
    "\nDuplicate question IDs:",
    evaluation_df["question_id"].duplicated().sum()
)

# Check for missing values
print(
    "\nMissing values:",
    evaluation_df.isnull().sum().sum()
)

Total questions: 80

Questions by category:
category
Factual                   20
Conceptual                20
Multi-paper Comparison    20
Unanswerable              20
Name: count, dtype: int64

Answerable distribution:
answerable
True     60
False    20
Name: count, dtype: int64

Duplicate question IDs: 0

Missing values: 0


## Configuration A – Standalone LLM

This configuration evaluates the performance of the Large Language Model (LLM) without using an external retrieval system.

The model receives each evaluation question directly and generates an answer using its pretrained knowledge.

This configuration serves as the baseline for comparison with the retrieval-augmented configurations.

The evaluation will examine answer correctness, relevance, faithfulness, citation validity, abstention behaviour, and response time.

## Inspect Existing Model and Evaluation Setup

Before evaluating Configuration A, we inspect the current notebook environment to identify any previously loaded language model, tokenizer, and relevant evaluation functions.

This helps us reuse existing components and avoid unnecessary model loading or variable conflicts.

In [13]:
# Inspect existing model, tokenizer, and evaluation-related variables
# This cell only checks the current environment; it does not modify anything.

keywords = [
    "model",
    "tokenizer",
    "pipeline",
    "generate",
    "retriever",
    "search",
    "evaluate"
]

print("EXISTING MODEL AND EVALUATION COMPONENTS")
print("-" * 50)

found_components = False

for name, obj in list(globals().items()):

    # Look for relevant variable names
    if any(keyword in name.lower() for keyword in keywords):

        # Ignore imported modules and internal notebook variables
        if name.startswith("_"):
            continue

        found_components = True

        print(f"\nVariable name: {name}")
        print(f"Object type: {type(obj).__name__}")

        # Display model information if available
        config = getattr(obj, "config", None)

        if config is not None:
            model_name = getattr(config, "_name_or_path", None)

            if model_name:
                print(f"Model identifier: {model_name}")

        # Display tokenizer information if available
        tokenizer_name = getattr(obj, "name_or_path", None)

        if tokenizer_name:
            print(f"Tokenizer identifier: {tokenizer_name}")

if not found_components:
    print("No matching components found in the current notebook environment.")

print("\nInspection completed.")

EXISTING MODEL AND EVALUATION COMPONENTS
--------------------------------------------------

Variable name: MODEL_PATH
Object type: PosixPath

Variable name: generated_answer
Object type: str

Variable name: tokenizer_name
Object type: NoneType

Inspection completed.


## Load Qwen Model for Evaluation

The Qwen2.5-0.5B-Instruct model was downloaded and configured in the original project notebook.

Since this is a new evaluation notebook, the model must be loaded again into the current Python session.

The previously downloaded GGUF model file will be reused. There is no need to download it again.

In [3]:
from pathlib import Path
from llama_cpp import Llama

# Locate the project directory
PROJECT_DIR = Path.cwd()

# Set the path to the downloaded Qwen model
MODEL_PATH = (
    PROJECT_DIR / "models" /
    "qwen2.5-0.5b-instruct-q4_k_m.gguf"
)

# Check whether the model file exists
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Qwen model file not found: {MODEL_PATH}"
    )

# Load the Qwen language model
llm = Llama(
    model_path=str(MODEL_PATH),
    n_ctx=1024,
    n_batch=128,
    n_threads=4,
    n_gpu_layers=0,
    verbose=False
)

print("Qwen model loaded successfully!")
print("Model location:", MODEL_PATH)
print("Model ready for evaluation.")

Qwen model loaded successfully!
Model location: /Users/macbookair/DAT401_Research_Assistant/models/qwen2.5-0.5b-instruct-q4_k_m.gguf
Model ready for evaluation.


In [4]:
import time
import pandas as pd

# Store the generated responses
standalone_results = []

# System prompt for the standalone LLM
system_prompt = """
You are a research assistant.

Answer the user's question using only your existing knowledge.
No research papers or supporting documents are provided.

Do not invent research findings, statistics, study details,
or citations.

If you do not have enough reliable information to answer,
clearly state that you do not have enough information.

Keep your answers clear, concise, and relevant.
"""

# Generate an answer for every evaluation question
for _, row in evaluation_df.iterrows():

    question = row["question"]

    # Record the generation start time
    start_time = time.perf_counter()

    # Ask Qwen to generate an answer
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt
            },
            {
                "role": "user",
                "content": question
            }
        ],
        max_tokens=150,
        temperature=0.2
    )

    # Extract the generated answer
    generated_answer = (
        response["choices"][0]["message"]["content"].strip()
    )

    # Calculate response time
    response_time = time.perf_counter() - start_time

    # Store the question and its generated response
    standalone_results.append({
        "question_id": row["question_id"],
        "category": row["category"],
        "question": question,
        "gold_answer": row["gold_answer"],
        "answerable": row["answerable"],
        "supporting_sources": row["supporting_sources"],
        "generated_answer": generated_answer,
        "response_time": response_time
    })

    # Display progress
    print(f"Completed {row['question_id']}")

# Convert results into a DataFrame
standalone_df = pd.DataFrame(standalone_results)

print("\nStandalone LLM answer generation completed!")
print("Total responses:", len(standalone_df))

Completed Q001
Completed Q002
Completed Q003
Completed Q004
Completed Q005
Completed Q006
Completed Q007
Completed Q008
Completed Q009
Completed Q010
Completed Q011
Completed Q012
Completed Q013
Completed Q014
Completed Q015
Completed Q016
Completed Q017
Completed Q018
Completed Q019
Completed Q020
Completed Q021
Completed Q022
Completed Q023
Completed Q024
Completed Q025
Completed Q026
Completed Q027
Completed Q028
Completed Q029
Completed Q030
Completed Q031
Completed Q032
Completed Q033
Completed Q034
Completed Q035
Completed Q036
Completed Q037
Completed Q038
Completed Q039
Completed Q040
Completed Q041
Completed Q042
Completed Q043
Completed Q044
Completed Q045
Completed Q046
Completed Q047
Completed Q048
Completed Q049
Completed Q050
Completed Q051
Completed Q052
Completed Q053
Completed Q054
Completed Q055
Completed Q056
Completed Q057
Completed Q058
Completed Q059
Completed Q060
Completed Q061
Completed Q062
Completed Q063
Completed Q064
Completed Q065
Completed Q066
Completed 

## Inspection of Standalone LLM Responses

After generating answers for all 80 evaluation questions, the responses must be inspected to ensure that they have been stored correctly.

This step checks:
- The total number of generated responses.
- Missing or empty generated answers.
- The average response time.
- Sample responses from answerable and unanswerable questions.

This inspection helps verify the generated results before proceeding to performance evaluation.

In [5]:
# Check the total number of generated responses
print("Total generated responses:", len(standalone_df))

# Check for missing answers
print(
    "\nMissing generated answers:",
    standalone_df["generated_answer"].isnull().sum()
)

# Check for empty answers
empty_answers = (
    standalone_df["generated_answer"]
    .str.strip()
    .eq("")
    .sum()
)

print("Empty generated answers:", empty_answers)

# Calculate average response time
print(
    "\nAverage response time:",
    round(standalone_df["response_time"].mean(), 3),
    "seconds"
)

# Display five sample responses from answerable questions
print("\nSample responses: Answerable questions")

display(
    standalone_df[standalone_df["answerable"] == True][
        ["question_id", "question", "gold_answer", "generated_answer"]
    ].head(5)
)

# Display five sample responses from unanswerable questions
print("\nSample responses: Unanswerable questions")

display(
    standalone_df[standalone_df["answerable"] == False][
        ["question_id", "question", "generated_answer"]
    ].head(5)
)

Total generated responses: 80

Missing generated answers: 0
Empty generated answers: 0

Average response time: 1.108 seconds

Sample responses: Answerable questions


,question_id,question,gold_answer,generated_answer
0,Q001,What machine learning methods are used for bra...,Research on brain tumour detection uses tradit...,Brain tumour detection using machine learning ...
1,Q002,What deep learning architectures are used for ...,Convolutional Neural Networks (CNNs) are used ...,Deep learning architectures are commonly used ...
2,Q003,What type of MRI images was used to collect th...,The study used spin-echo-weighted MRI images t...,The study used MRI images to collect the menin...
3,Q004,How many brain tumour categories were included...,The dataset included six brain tumour categori...,The multiclass MRI dataset includes 12 brain t...
4,Q005,What classification accuracy did the suggested...,The suggested CNN achieved 96.13% accuracy on ...,"I'm sorry, but I cannot provide an answer to y..."



Sample responses: Unanswerable questions


,question_id,question,generated_answer
60,Q061,What was the five-year survival rate of brain ...,I do not have enough reliable information to a...
61,Q062,How many patients were diagnosed with brain tu...,I do not have enough information to answer you...
62,Q063,Which hospital in Bhutan provided the MRI scan...,"I'm sorry, but I do not have enough informatio..."
63,Q064,What was the total financial cost of developin...,"I'm sorry, but I don't have enough information..."
64,Q065,Which medical expert or neurologist supervised...,I do not have enough information to answer you...


## Save Standalone LLM Evaluation Results

The generated responses from Configuration A are saved for subsequent evaluation and comparison.

Saving these results allows the generated answers, reference answers, answerability labels, and response times to be reused without running the model again.

The results will be stored in CSV format inside the evaluation folder.